# Application Scenario 4: Vehicle Detection

This scenario exports the output to the `output` directory.\
Please clear this directory between pipeline executions.

## Download the resources

- configuration file
- input directory containing the GeoJSON file

In [ ]:
!curl -sL https://raw.githubusercontent.com/geospaitial-lab/aviary-ACM-SIGSPATIAL-2026/main/application_scenarios/4/config.yaml -o config.yaml
!mkdir output
!mkdir input
!mkdir my_plugin
!curl -sL https://raw.githubusercontent.com/geospaitial-lab/aviary-ACM-SIGSPATIAL-2026/main/application_scenarios/4/input/boundary.geojson -o input/boundary.geojson
!curl -sL https://raw.githubusercontent.com/geospaitial-lab/aviary-ACM-SIGSPATIAL-2026/main/application_scenarios/4/my_plugin/__init__.py -o my_plugin/__init__.py
!curl -sL https://raw.githubusercontent.com/geospaitial-lab/aviary-ACM-SIGSPATIAL-2026/main/application_scenarios/4/my_plugin/vehicle_model.py -o my_plugin/vehicle_model.py

## Installation

### With `pip`

In [ ]:
!pip install geospaitial-lab-aviary[cli] ipywidgets ultralytics

### With `uv`

In [1]:
!uv pip install geospaitial-lab-aviary[cli] ipywidgets ultralytics

Using Python 3.13.0 environment at C:\Users\mrsmrynk\PycharmProjects\aviary-ACM-SIGSPATIAL-2026\.venv
Audited 3 packages in 22ms


---

## Python API

### Imports

In [2]:
import warnings
from pathlib import Path

import aviary
import aviary.pipeline
import aviary.tile
import aviary.utils
import aviary.vector
import geopandas as gpd

from my_plugin import VehicleModel

### Check the version

In [3]:
print(aviary.__version__)

1.9.1


### Suppress experimental warnings

Some components used in this scenario are experimental.

In [4]:
warnings.filterwarnings('ignore', category=aviary.AviaryExperimentalWarning)

### Define the `Grid`

[Docs](https://geospaitial-lab.github.io/aviary/api_reference/core/grid)

In [15]:
gdf = gpd.read_file('input/boundary.geojson')
gdf = gdf.to_crs('EPSG:25832')

grid = aviary.Grid.from_gdf(
    gdf=gdf,
    tile_size=100,
)

grid = grid.chunk(num_chunks=500)[93]

Grid(
    coordinates=22,
    tile_size=100,
)


### Define the `TileFetcher`

[Docs](https://geospaitial-lab.github.io/aviary/api_reference/tile/tile_fetcher/tile_fetcher)

In [6]:
wms_fetcher = aviary.tile.WMSFetcher(
    url='https://www.wms.nrw.de/geobasis/wms_nw_dop',
    version=aviary.WMSVersion.V1_3_0,
    layer='nw_dop_rgb',
    epsg_code=25832,
    response_format='image/png',
    channel_names=['r', 'g', 'b'],
    tile_size=100,
    ground_sampling_distance=.2,
    buffer_size=20,
)

### Define the `TilesProcessor`

[Docs](https://geospaitial-lab.github.io/aviary/api_reference/tile/tiles_processor/tiles_processor)

In [7]:
vehicle_model = VehicleModel(
    r_channel_name='r',
    g_channel_name='g',
    b_channel_name='b',
    vehicle_channel_name='vehicle',
)

remove_buffer_processor = aviary.tile.RemoveBufferProcessor()

object_exporter = aviary.tile.ObjectExporter(
    channel_name='vehicle',
    epsg_code=25832,
    path=Path('output/vehicle.gpkg'),
    mode=aviary.ObjectExporterMode.POINT,
)

grid_exporter = aviary.tile.GridExporter(
    path=Path('output/processed.json'),
)

composite_processor = aviary.tile.SequentialCompositeProcessor(
    tiles_processors=[
        vehicle_model,
        remove_buffer_processor,
        object_exporter,
        grid_exporter,
    ],
)

### Define the `TilePipeline`

[Docs](https://geospaitial-lab.github.io/aviary/api_reference/pipeline/pipeline)

In [8]:
tile_pipeline = aviary.pipeline.TilePipeline(
    grid=grid,
    tile_fetcher=wms_fetcher,
    tiles_processor=composite_processor,
    tile_loader_batch_size=1,
    tile_loader_max_num_threads=None,
    tile_loader_num_prefetched_tiles=0,
    show_progress=True,
)

### Define the `VectorLoader`

[Docs](https://geospaitial-lab.github.io/aviary/api_reference/vector/vector_loader/vector_loader)

In [9]:
gpkg_loader = aviary.vector.GPKGLoader(
    path=Path('output/vehicle.gpkg'),
    epsg_code=25832,
    layer_name='vehicle',
)

geojson_loader = aviary.vector.GeoJSONLoader(
    path=Path('input/boundary.geojson'),
    epsg_code=25832,
    layer_name='boundary',
)

composite_loader = aviary.vector.CompositeLoader(
    vector_loaders=[gpkg_loader, geojson_loader],
)

### Define the `VectorProcessor`

[Docs](https://geospaitial-lab.github.io/aviary/api_reference/vector/vector_processor/vector_processor)

In [10]:
clip_processor = aviary.vector.ClipProcessor(
    layer_name='vehicle',
    mask_layer_name='boundary',
)

query_processor = aviary.vector.QueryProcessor(
    layer_name='vehicle',
    query_string='score > 0.5',
)

map_field_processor = aviary.vector.MapFieldProcessor(
    layer_name='vehicle',
    field='value',
    mapping={
        9: 'Large vehicle',
        10: 'Small vehicle',
    }
)

vector_exporter_ = aviary.vector.VectorExporter(
    layer_name='vehicle',
    epsg_code=25832,
    path=Path('output/vehicle.gpkg'),
)

composite_processor_ = aviary.vector.SequentialCompositeProcessor(
    vector_processors=[
        clip_processor,
        query_processor,
        map_field_processor,
        vector_exporter_,
    ]
)

### Define the `VectorPipeline`

[Docs](https://geospaitial-lab.github.io/aviary/api_reference/pipeline/pipeline)

In [11]:
vector_pipeline = aviary.pipeline.VectorPipeline(
    vector_loader=composite_loader,
    vector_processor=composite_processor_,
)

### Define the `Logger`

[Docs](https://geospaitial-lab.github.io/aviary/api_reference/utils/logger)

In [12]:
logger = aviary.utils.Logger(
    sink='output/application_scenario_4.log',
    level=aviary.LogLevel.TRACE,
)

### Define the `CompositePipeline`

[Docs](https://geospaitial-lab.github.io/aviary/api_reference/pipeline/pipeline)

In [13]:
composite_pipeline = aviary.pipeline.CompositePipeline(
    pipelines=[tile_pipeline, vector_pipeline]
)

### Execute the `CompositePipeline`

In [14]:
composite_pipeline()

Output()

AviaryUserError: Invalid coordinates! The coordinates must be evenly distributed.

---

## CLI

### Check the version

In [ ]:
!aviary --version

### Suppress experimental warnings

Some components used in this scenario are experimental.

In [ ]:
%env AVIARY_EXPERIMENTAL_WARNINGS=false

### Validate the configuration file

[Docs](https://geospaitial-lab.github.io/aviary/cli_reference/aviary_pipeline/pipeline_validate)

In [ ]:
!aviary pipeline validate config.yaml

### Set the path to the log file and the log level

In [ ]:
%env AVIARY_LOG_PATH=output/application_scenario_4.log
%env AVIARY_LOG_LEVEL=trace

### Execute the `CompositePipeline`

[Docs](https://geospaitial-lab.github.io/aviary/cli_reference/aviary_pipeline/pipeline_run)

In [ ]:
!aviary pipeline run config.yaml